# 01 · Data acquisition

**LAEI 2019 · Module End Project**

Downloads the London Atmospheric Emissions Inventory 2019 from the London
Datastore, unzips it, and converts the three source workbooks into flat CSVs.

**Run this once.** Every cell is idempotent — re-running skips work already
done — but the full cold-start cost is roughly:

| Stage | Cost |
|---|---|
| Download | ~445 MB |
| Unzip | ~490 MB on disk |
| Excel → CSV conversion | **~18 minutes of compute** |

The conversion is the expensive part and is unavoidable: the road-link
workbook is 345 MB of XML and expands to many gigabytes if opened with
`pandas.read_excel`. It is streamed once with `openpyxl` and never touched
again.

**Source.** London Datastore dataset `e758q`, published by the GLA Environment
Team under the UK Open Government Licence v3.0.

In [1]:
import sys, zipfile, time
from pathlib import Path
import requests

sys.path.append("../src")

ROOT    = Path("..").resolve()
RAW     = ROOT / "data" / "raw"
INTERIM = ROOT / "data" / "interim"
PROC    = ROOT / "data" / "processed"
for d in (RAW, INTERIM, PROC):
    d.mkdir(parents=True, exist_ok=True)

print("project root:", ROOT)

project root: C:\Users\Sambo\OneDrive\Documents\University of Liverpool\Modules\Module 3\ModuleEndProject


## 1 · Download

These direct-download URLs were verified working against the Datastore API.
Three files drive every model; the fourth (GIS boundaries) is small and makes
the presentation maps possible.

The concentration grids — 536 MB CSV, 393 MB ASCII, 392 MB GIS — are
**deliberately not downloaded**. They hold modelled ground-level
concentrations at 20 m resolution, which is out of scope: this project models
*emissions* (what boroughs can influence), not *concentrations* (which
additionally depend on meteorology and dispersion).

In [2]:
BASE = "https://data.london.gov.uk/download/e758q"

FILES = {
    # --- required ---
    "LAEI-2019-Emissions-Summary-including-Forecast.zip":
        f"{BASE}/17d21cd1-892e-4388-9fea-b48c1b61ee3c/"
        "LAEI-2019-Emissions-Summary-including-Forecast.zip",
    "laei-2019-major-roads-vkm-flows-speeds.zip":
        f"{BASE}/3a00296b-c88b-4de0-8336-b127034cc07b/"
        "laei-2019-major-roads-vkm-flows-speeds.zip",
    "LAEI2019-nox-pm-co2-major-roads-link-emissions.zip":
        f"{BASE}/f2129345-06d4-4d3d-8304-d65fcdeb118e/"
        "LAEI2019-nox-pm-co2-major-roads-link-emissions.zip",
    # --- small, for maps in the presentation ---
    "LAEI2019-Supporting-Information-GIS-Files.zip":
        f"{BASE}/f2e992aa-2fed-4380-af99-99fc98f3853b/"
        "LAEI2019-Supporting-Information-GIS-Files.zip",
}

for name, url in FILES.items():
    dest = RAW / name
    if dest.exists():
        print(f"  skip (have) {name:<58} {dest.stat().st_size/1e6:7.1f} MB")
        continue
    print(f"  downloading {name} ...", flush=True)
    t0 = time.time()
    with requests.get(url, stream=True, timeout=600) as r:
        r.raise_for_status()
        with open(dest, "wb") as f:
            for chunk in r.iter_content(1 << 20):
                f.write(chunk)
    print(f"    -> {dest.stat().st_size/1e6:.1f} MB in {time.time()-t0:.0f}s")

print(f"\ntotal in data/raw: {sum(p.stat().st_size for p in RAW.glob('*'))/1e6:,.0f} MB")

  skip (have) LAEI-2019-Emissions-Summary-including-Forecast.zip           101.0 MB
  skip (have) laei-2019-major-roads-vkm-flows-speeds.zip                    23.5 MB
  skip (have) LAEI2019-nox-pm-co2-major-roads-link-emissions.zip           343.1 MB
  skip (have) LAEI2019-Supporting-Information-GIS-Files.zip                  2.7 MB

total in data/raw: 490 MB


## 2 · Unzip

In [3]:
for z in sorted(RAW.glob("*.zip")):
    with zipfile.ZipFile(z) as zf:
        members = zf.namelist()
        missing = [m for m in members if not (INTERIM / m).exists()]
        if not missing:
            print(f"  skip (have) {z.name}")
            continue
        print(f"  extracting {z.name}  ({len(missing)} member(s))")
        zf.extractall(INTERIM)

for p in sorted(INTERIM.glob("*.xlsx")):
    print(f"  {p.name:<58} {p.stat().st_size/1e6:7.1f} MB")

  skip (have) LAEI-2019-Emissions-Summary-including-Forecast.zip
  skip (have) laei-2019-major-roads-vkm-flows-speeds.zip
  skip (have) LAEI2019-nox-pm-co2-major-roads-link-emissions.zip
  extracting LAEI2019-nox-pm-co2-minor-roads-grid-emissions.zip  (1 member(s))
  extracting LAEI2019-nox-pm-cold-start-grid-emissions.zip  (1 member(s))


  extracting LAEI2019-Supporting-Information-GIS-Files.zip  (59 member(s))


  LAEI-2019-Emissions-Summary-including-Forecast.xlsx          119.9 MB
  laei-2019-major-roads-vkm-flows-speeds.xlsx                   23.5 MB
  LAEI2019-nox-pm-co2-major-roads-link-emissions.xlsx          345.4 MB
  LAEI2019-nox-pm-co2-minor-roads-grid-emissions.xlsx           16.2 MB
  LAEI2019-nox-pm-cold-start-grid-emissions.xlsx                 3.2 MB


## 3 · Convert Excel → CSV

Delegated to `src/convert_excel.py` so the modelling notebooks and this
notebook cannot drift apart. That module handles the three source-specific
hazards documented in notebook 02:

1. the workbooks declare a bogus `1 × 1` sheet dimension, so
   `ws.reset_dimensions()` is mandatory;
2. the PM10 / PM2.5 sheets hold **three rows per link** (exhaust, brake wear,
   tyre wear) and must be **summed**, not overwritten;
3. headers carry leading and trailing whitespace.

Row counts are asserted inside the module — if a count is wrong it exits
non-zero rather than writing a quietly corrupt CSV.

In [4]:
EXPECTED = {
    "grid_all_years.csv": 699_120,
    "link_features.csv":   79_437,
    "link_targets.csv":    79_439,
}

have = {n: (PROC / n).exists() for n in EXPECTED}
print("already converted:", {k: v for k, v in have.items()})

if all(have.values()):
    print("\nAll three CSVs present — skipping the ~18 minute conversion.")
    print("Delete a file from data/processed/ to force a rebuild.")
else:
    print("\nRunning src/convert_excel.py (this takes ~18 minutes) ...\n", flush=True)
    import subprocess
    r = subprocess.run([sys.executable, str(ROOT / "src" / "convert_excel.py")],
                       capture_output=True, text=True)
    print(r.stdout)
    if r.returncode != 0:
        print(r.stderr)
        raise RuntimeError("conversion failed")

already converted: {'grid_all_years.csv': True, 'link_features.csv': True, 'link_targets.csv': True}

All three CSVs present — skipping the ~18 minute conversion.
Delete a file from data/processed/ to force a rebuild.


## 4 · Verify

A hard check, not a glance. If any of these fails, stop and fix it before
touching a model — every downstream row count depends on them.

In [5]:
ok = True
for name, expected in EXPECTED.items():
    path = PROC / name
    n = sum(1 for _ in open(path, encoding="utf-8")) - 1   # minus header
    status = "OK" if n == expected else f"MISMATCH (expected {expected:,})"
    ok &= (n == expected)
    print(f"  {name:<24} {n:>9,} rows  {path.stat().st_size/1e6:7.1f} MB  {status}")

assert ok, "row-count verification failed"
print("\nAll row counts verified.")

  grid_all_years.csv         699,120 rows    128.7 MB  OK
  link_features.csv           79,437 rows     19.0 MB  OK
  link_targets.csv            79,439 rows      7.8 MB  OK

All row counts verified.


In [6]:
import pandas as pd

# spot-check the PM accumulation: PM2.5 is a subset of PM10, so the ratio
# must exceed 1 but stay physically plausible (roughly 1.5-2.5 for road
# transport, where brake and tyre wear dominate the coarse fraction).
t = pd.read_csv(PROC / "link_targets.csv")
ratio = t["pm10"].sum() / t["pm25"].sum()
print(f"major-road totals (tonnes/yr, 2019)")
print(f"  NOx   {t['nox'].sum():>12,.1f}")
print(f"  PM10  {t['pm10'].sum():>12,.1f}")
print(f"  PM2.5 {t['pm25'].sum():>12,.1f}")
print(f"  CO2   {t['co2'].sum():>12,.0f}")
print(f"\n  PM10 / PM2.5 ratio = {ratio:.2f}", end="  ")
print("(plausible)" if 1.2 < ratio < 3 else "<-- INVESTIGATE: PM may not be summed correctly")

g = pd.read_csv(PROC / "grid_all_years.csv", low_memory=False, usecols=["Year", "Grid ID 2019", "nox"])
d = g[g.Year == 2019]
print(f"\ngrid 2019: {len(d):,} rows over {d['Grid ID 2019'].nunique():,} cells, "
      f"NOx {d['nox'].sum():,.0f} t/yr")

major-road totals (tonnes/yr, 2019)
  NOx       19,446.6
  PM10       2,239.7
  PM2.5      1,157.0
  CO2      8,784,860

  PM10 / PM2.5 ratio = 1.94  (plausible)



grid 2019: 143,976 rows over 3,460 cells, NOx 45,405 t/yr


## Done

`data/processed/` now holds the three flat CSVs. Continue with
**`02_eda_and_profiling.ipynb`**, which profiles them and writes the prepared
modelling tables.

> **Provenance note for the report.** LAEI 2019 was **superseded by LAEI 2022
> in August 2025**. It remains published and is the dataset this brief
> specifies, but the supersession should be acknowledged — and the pipeline
> built here is deliberately re-pointable at the newer release.